**TODO:明天开始从头写，一点点替代PyTorch**

In [1]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST

train_data = MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(batch_size=64, shuffle=True, num_workers=0, dataset=train_data)

In [ ]:
import numpy as np

# ============================================================
# 第一题：ReLU
# ============================================================
# forward: 输入 x，输出 np.maximum(0, x)
#   保存 x 用于 backward
# backward: 输入 dL_dZ（本层输出的梯度）
#   输出: dL_dZ * (x > 0)  ← 只有正向时 >0 的神经元才传梯度
#   （你需要在 __init__ 里存 forward 的输入）
#   （不需要存其他东西，去掉无用的属性）


class ReLU:
    def __init__(self):
        # TODO: 初始化需要的缓存变量
        pass

    def forward(self, x):
        # TODO: 保存 x，返回 ReLU 结果
        pass

    def backward(self, dL_dZ):
        # TODO: 用 forward 时保存的 x 来过滤梯度
        pass


# ============================================================
# 第二题：Linear
# ============================================================
# forward: Z = A_prev @ W + b
#   形状: A_prev (N, in_dim), W (in_dim, out_dim), b (out_dim,)
#   保存 A_prev 用于 backward 的 dW 计算
#
# backward: 输入 dL_dZ (N, out_dim) ← loss 对本层输出的梯度
#   你要算三样东西:
#     1. dW = ???  (形状必须 = W 的形状 = (in_dim, out_dim))
#        回忆: C[i,j] = Σ A[i,k]·B[k,j]，那 dL/dW 需要 A_prev 的行和 dL_dZ 的列怎么组合？
#        提示: 谁转置放前面可以让 A_prev 的 in_dim 维度和 dL_dZ 的 out_dim 维度凑出 (in_dim, out_dim)？
#     2. db = np.sum(dL_dZ, axis=0)  ← 已经给你了，b 是 (out_dim,)，沿 batch 维求和
#     3. 返回 dL_dZ @ W.T  ← 传给上一层的梯度，形状 (N, in_dim)
#        为什么是 W.T？因为 Z = A_prev @ W，Z 对 A_prev 的偏导是 W^T
#
# step: W -= lr * dW, b -= lr * db


class Linear:
    def __init__(self, in_dim, out_dim, lr=0.001):
        self.W = np.random.randn(in_dim, out_dim) * lr
        self.b = np.zeros(out_dim)
        self.lr = lr

        # TODO: 初始化 forward 和 backward 需要的缓存变量
        pass

    def forward(self, A_prev):
        # TODO: 保存 A_prev，计算并返回 Z = A_prev @ W + b
        pass

    def backward(self, dL_dZ):
        # dL_dZ 形状: (N, out_dim)
        # TODO: 计算 dW（形状 (in_dim, out_dim)）
        # TODO: 计算 db（形状 (out_dim,)）
        # TODO: 返回传给上一层的梯度（形状 (N, in_dim)）
        pass

    def step(self):
        # TODO: 用 dW 和 db 更新 W 和 b
        pass


# ============================================================
# 第三题：CrossEntropyLoss
# ============================================================
# forward: 输入 A (logits, 形状 (N, num_classes)) 和 t (标签)
#   1. softmax: 对 A 的每一行单独做
#      稳定版: A_stable = A - A.max(axis=1, keepdims=True)
#              A_exp = exp(A_stable)
#              A_softmax = A_exp / A_exp.sum(axis=?, keepdims=True)
#      思考: softmax 应该沿 axis=0 还是 axis=1？
#   2. one-hot: 如果 t 是一维的 (N,)，转成 one-hot (N, num_classes)
#      注意: 不要覆盖原始标签！先用新变量存 one-hot，最后再赋值
#   3. loss: -sum(t_onehot * log(A_softmax + 1e-12)) / N
#      返回 loss 值
#
# backward: 输入: 无（用 forward 保存的变量）
#   交叉熵 + softmax 的联合梯度 = (A_softmax - t_onehot) / N
#   返回的形状应该是 (N, num_classes)，不要转置！


class CrossEntropyLoss:
    def __init__(self):
        # TODO: 初始化需要的缓存变量
        pass

    def forward(self, A, t):
        # A: (N, num_classes), t: (N,) 或 (N, num_classes)
        # TODO: 1. 稳定版 softmax
        # TODO: 2. 转 one-hot（如果 t 是一维的）
        # TODO: 3. 计算交叉熵 loss
        # TODO: 4. 保存 backward 需要的变量
        # TODO: 5. 返回 loss 值
        pass

    def backward(self):
        # TODO: 返回交叉熵 + softmax 的联合梯度
        pass


In [ ]:
# ============================================================
# 第四题：训练循环
# ============================================================
# 1. 搭模型: Linear(784, 128) → ReLU → Linear(128, 10)
#    MNIST 图是 28×28 = 784 像素，10 个数字类别
#    学习率建议 0.01
#
# 2. 训练 5 个 epoch，每个 epoch:
#    - 遍历 train_loader，拿到 x 和 y
#    - x 是 (batch, 1, 28, 28)，用 x.view(x.shape[0], -1).numpy() 展平成 (batch, 784)
#    - y 是 (batch,) 的标签
#
#    forward:
#    - 让 x 依次通过所有 layer 的 forward
#    - 把最终输出传给 loss_fn.forward()
#
#    backward:
#    - loss_fn.backward() 得到初始梯度
#    - 倒序遍历 layers（reversed(layers)），把梯度传给每一层的 backward，每次更新 grad
#
#    step:
#    - 遍历 layers，对每个 Linear 实例调 step()
#
#    - 累计 loss 和 accuracy（argmax(axis=1) == y 的准确率）
#
# 3. 每 epoch 打印: Loss 和 Accuracy
#
# 期望: 5 epoch 后 accuracy ~95%

layers = [
    # TODO: Linear(784, 128, lr=0.01)
    # TODO: ReLU()
    # TODO: Linear(128, 10, lr=0.01)
]
loss_fn = None  # TODO: CrossEntropyLoss()

# TODO: 训练循环
